# Text 생성 및 Prompt 예제

### Responses API
| 기능 | **특징** |
| --- | --- |
| **상태 유지(Stateful)** | ✅ 서버 측에서 **대화 상태를 관리(저장 및 이어쓰기 가능)** |
| **내장 도구 지원(Built-in Tools)** | ✅ `web_search`, `file_search`, `computer_use_preview` 등 자동 호출 가능 |
| **구조 & 사용 방식** | `instructions`, `input`, `tools`, `store`, `previous_response_id` 기반 API |
| **응답 형식 (Output)** | `response.output_text`, `response.output` 구조로 단일 메세지 응답 및 도구 호출 결과 포함 |
| **사용 목적** | ✅ **에이전트/도구 사용, 파일/웹 검색, 상태 관리**가 필요한 복잡한 워크플로우에 적합 |
| **미래 확장성** | 🔧 OpenAI Roadmap 중심 – Chat API의 기능을 통합 발전시키는 방향 |유지                                    |

[1]: https://simonwillison.net/2025/Mar/11/responses-vs-chat-completions/?utm_source=chatgpt.com "OpenAI API: Responses vs. Chat Completions"
[2]: https://cookbook.openai.com/examples/responses_api/responses_example?utm_source=chatgpt.com "Web Search and States with Responses API - OpenAI Cookbook"


In [1]:
import os
import openai
import sys
sys.path.append('./')

from dotenv import load_dotenv
load_dotenv() # read local .env file

True

In [2]:
from openai import OpenAI
client = OpenAI()

models = client.models.list()

model_names = [m.id for m in models.data]
print(model_names)

['text-embedding-ada-002', 'whisper-1', 'gpt-3.5-turbo', 'tts-1', 'gpt-3.5-turbo-16k', 'gpt-4-0613', 'gpt-4', 'davinci-002', 'babbage-002', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo-instruct-0914', 'gpt-3.5-turbo-1106', 'tts-1-hd', 'tts-1-1106', 'tts-1-hd-1106', 'text-embedding-3-small', 'text-embedding-3-large', 'gpt-3.5-turbo-0125', 'gpt-4-turbo', 'gpt-4-turbo-2024-04-09', 'gpt-4o', 'gpt-4o-2024-05-13', 'gpt-4o-mini-2024-07-18', 'gpt-4o-mini', 'gpt-4o-2024-08-06', 'omni-moderation-latest', 'omni-moderation-2024-09-26', 'o1-2024-12-17', 'o1', 'computer-use-preview', 'o3-mini', 'o3-mini-2025-01-31', 'gpt-4o-2024-11-20', 'computer-use-preview-2025-03-11', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4o-mini-search-preview', 'gpt-4o-transcribe', 'gpt-4o-mini-transcribe', 'o1-pro-2025-03-19', 'o1-pro', 'gpt-4o-mini-tts', 'o3-2025-04-16', 'o4-mini-2025-04-16', 'o3', 'o4-mini', 'gpt-4.1-2025-04-14', 'gpt-4.1', 'gpt-4.1-mini-2025-04-14', 'gpt-4.1-mini', 'gpt-4.1-nano-2025-04-14', 'gpt-4.1-na

In [3]:
Model = "gpt-5-nano"

## Responses API

## 문법 수정

SYSTEM : 당신은 문장을 받게 될 것이며, 당신의 임무는 그것을 표준 한국어로 변환하는 것입니다.  
USER :   
안갔어 시장에 그녀는.

In [4]:
# Responses API 호출 예시
response = client.responses.create(
    model=Model,  
    instructions="당신은 문장을 받게 될 것이며, 당신의 임무는 그것을 표준 한국어로 변환하는 것입니다.",
    input="안갔어 시장에 그녀는."
)

# 응답 결과 출력
print(response.output_text)

그녀는 시장에 가지 않았다.


## 구조화되지 않은 데이터의 구문 분석
SYSTEM : 구조화되지 않은 데이터가 제공되며 이를 CSV 형식으로 구문 분석하는 작업이 수행됩니다.  
USER :   
최근 발견된 행성 구크럭스(Goocrux)에서는 많은 과일이 발견됐다. 그곳에서 자라는 네오스키즐이 있는데, 보라색이고 사탕 맛이 납니다. 회색 빛이 도는 파란색 과일이고 매우 시큼하며 레몬과 약간 비슷한 로헤클(loheckles)도 있습니다. 포유닛은 밝은 녹색을 띠며 단맛보다 풍미가 더 좋습니다. 네온 핑크색 맛과 솜사탕 같은 맛이 나는 루프노바도 많이 있습니다. 마지막으로 글로울(glowls)이라는 과일이 있는데, 이 과일은 신맛과 부식성이 있는 매우 신맛과 쓴맛이 나며 옅은 오렌지색을 띠고 있습니다.

In [5]:
TEXT = """
      최근 발견된 행성 구크럭스(Goocrux)에서는 많은 과일이 발견됐다. 그곳에서 자라는 네오스키즐이 있는데, 보라색이고 사탕 맛이 납니다. 
      회색 빛이 도는 파란색 과일이고 매우 시큼하며 레몬과 약간 비슷한 로헤클(loheckles)도 있습니다. 포유닛은 밝은 녹색을 띠며 단맛보다 풍미가 더 좋습니다. 
      네온 핑크색 맛과 솜사탕 같은 맛이 나는 루프노바도 많이 있습니다. 마지막으로 글로울(glowls)이라는 과일이 있는데, 
      이 과일은 신맛과 부식성이 있는 매우 신맛과 쓴맛이 나며 옅은 오렌지색을 띠고 있습니다.
      이 데이터를 과일명, 색상, 맛으로 분석해 주세요.
      """

response = client.responses.create(
    model=Model,
    instructions="구조화되지 않은 데이터가 제공되며 이를 CSV 형식으로 구문 분석하는 작업이 수행됩니다.",
    input=TEXT
)

print(response.output_text)

과일명,색상,맛
네오스키즐,보라색,사탕 맛
로헤클,회색 빛이 도는 파란색,매우 시큼하고 레몬과 약간 비슷한 맛
포유닛,밝은 녹색,풍미가 강하고 단맛은 약함
루프노바,네온 핑크,"네온 핑크 맛과 솜사탕 맛"
글로울,옅은 오렌지색,"매우 신맛과 쓴맛"


In [6]:
from typing import List
from pydantic import BaseModel

# 개별 과일 정보 모델
class Fruit(BaseModel):
    name: str
    color: str
    taste: str

# 전체 응답 모델
class FruitList(BaseModel):
    fruits: List[Fruit]

response = client.responses.parse(
    model=Model,
    instructions=(
        "구조화되지 않은 문장에서 과일 정보를 추출하세요."
        "반드시 fruits 배열에 각 과일의 name, color, taste를 구조화해서 반환하세요."
    ),
    input=TEXT,
    text_format=FruitList,
)

# 파싱된 구조화 결과
parsed = response.output_parsed

for fruit in parsed.fruits:
    print(dict(fruit))

{'name': '네오스키즐', 'color': '보라색', 'taste': '사탕 맛'}
{'name': '로헤클(loheckles)', 'color': 'blue-gray', 'taste': '매우 시큼하고 레몬과 약간 비슷한'}
{'name': '포유닛', 'color': '밝은 녹색', 'taste': '강한 풍미(단맛 아님)'}
{'name': '루프노바', 'color': '네온 핑크', 'taste': '솜사탕 맛'}
{'name': '글로울(glowls)', 'color': '옅은 오렌지색', 'taste': '매우 신맛과 쓴맛'}


## 발화자 관리

| 역할       | 설명                                                                                      |
|------------|-------------------------------------------------------------------------------------------|
| developer  | 애플리케이션 개발자가 제공한 지침. **user 메시지보다 우선적으로 처리**됩니다.              |
| user       | 최종 사용자가 입력한 지침. developer 메시지보다 **우선순위가 낮습니다.**                  |
| assistant  | 모델이 생성한 응답 메시지. 응답 결과는 이 역할로 반환됩니다.                              |


In [7]:
response = client.responses.create(
    model=Model,
    input=[
        {"role": "developer", "content": "사용자에게 한국어로, 초보자도 이해하기 쉽게 5문장 이내로 설명하세요."},
        {"role": "user", "content": "안녕하세요."},
        {"role": "assistant", "content": "안녕하세요. 무었을 도와드릴까요?"},
        {"role": "user", "content": "딥러닝과 머신러닝의 차이점이 무었인가요?"},
    ],
)

print(response.output_text)

머신러닝은 데이터에서 패턴을 학습해 예측이나 의사결정을 하는 기술의 큰 범주예요. 그 안에 딥러닝이 속하는데, 딥러닝은 여러 층의 인공신경망을 사용해 데이터를 자동으로 특징으로 변환하고 학습합니다. 전통적인 머신러닝은 특징을 사람이 먼저 설계하거나 간단한 방법으로 추출하는 경우가 많지만, 딥러닝은 방대한 데이터와 계산으로 특징을 스스로 찾아내는 경향이 큽니다. 예로는 머신러닝의 의사결정 트리나 SVM 같은 알고리즘이 있고, 딥러닝은 이미지·음성·자연어 처리에 특히 강합니다. 데이터가 충분하고 컴퓨팅 파워가 많으면 딥러닝이 더 좋은 성능을 내지만, 데이터가 적으면 일반적인 머신러닝이 더 효율적일 수 있습니다.


## Keyword 추출 

In [8]:
text = """
"블랙 온 블랙 도자기(Black-on-Black ware)는 뉴멕시코 북부의 푸에블로 원주민 도자기 예술가들이 개발한 20세기 및 21세기 도자기 전통입니다. 
전통적인 환원 소성 블랙웨어는 푸에블로 예술가들에 의해 수세기 동안 만들어졌습니다. 
지난 세기의 흑색 자기는 표면이 매끄럽고 선택적 버니싱이나 내화 슬립을 적용하여 디자인을 적용한 제품입니다. 
또 다른 스타일은 디자인을 조각하거나 절개하고 융기된 부분을 선택적으로 연마하는 것입니다. 
여러 세대에 걸쳐 Kha'po Owingeh와 P'ohwhóge Owingeh 푸에블로의 여러 가족은 여주인 도예가들로부터 전수받은 기술을 사용하여 검은 바탕에 검은 도자기를 만들어 왔습니다. 
다른 푸에블로 출신의 예술가들도 검정색 바탕에 검정색 도자기를 제작했습니다. 몇몇 현대 예술가들은 조상의 도자기를 기리는 작품을 만들었습니다."
"""

In [9]:
response = client.responses.create(
    model=Model,
    instructions="텍스트 블록이 제공되며, 당신의 임무는 텍스트 블록에서 키워드 목록을 중요한 5 개만 추출하는 것입니다. 한글로 추출합니다.",
    input=text
)

In [10]:
print(response.output_text)

- 블랙 온 블랙 도자기
- 푸에블로
- 환원 소성 블랙웨어
- 버니싱
- Kha'po Owingeh


## Program Code 설명 

In [11]:
codes = """
class Log:
    def __init__(self, path):
        dirname = os.path.dirname(path)
        os.makedirs(dirname, exist_ok=True)
        f = open(path, "a+")

        # Check that the file is newline-terminated
        size = os.path.getsize(path)
        if size > 0:
            f.seek(size - 1)
            end = f.read(1)
            if end != "\n":
                f.write("\n")
        self.f = f
        self.path = path

    def log(self, event):
        event["_event_id"] = str(uuid.uuid4())
        json.dump(event, self.f)
        self.f.write("\n")

    def state(self):
        state = {"complete": set(), "last": None}
        for line in open(self.path):
            event = json.loads(line)
            if event["type"] == "submit" and event["success"]:
                state["complete"].add(event["id"])
                state["last"] = event
        return state
"""

In [12]:
response = client.responses.create(
    model=Model,
    instructions="당신에게는 코드 조각이 제공될 것이며, 무었을 하는 코드인지 한줄로 설명하세요. 한국어로 설명해 주세요.",
    input=codes
)

In [13]:
print(response.output_text)

파일에 한 줄씩 JSON 이벤트를 기록하고 각 이벤트에 _event_id를 부여해 저장하며, state()에서 성공한 submit 이벤트를 모아 마지막 이벤트를 반환하는 로깅 도구이다.


## 감성 분류기
- 한개의 text 감성 분석

In [14]:
response = client.responses.create(
    model=Model,
    instructions="당신은 텍스트를 입력 받게 될 것이고, 당신의 임무는 텍스트의 감정을 긍정적, 중립적, 부정적으로 분류하는 것입니다.",
    input="나는 새로운 배트맨 영화가 좋습니다!"
)

In [15]:
print(response.output_text)

positive


- 여러개의 text 감성 분석

In [16]:
sentiments = [
    "나는 새로운 배트맨 영화가 좋습니다!",
    "오늘 날씨가 너무 안 좋네요.",
    "커피 마시러 가는 길이 설레네요!",
    "회의가 너무 길고 지루했습니다.",
    "점심으로 먹은 피자가 맛있었어요."
]

responses = []
for sent in sentiments:
    response = client.responses.create(
        model=Model,
        instructions="당신은 텍스트를 입력 받게 될 것이고, 당신의 임무는 텍스트의 감정을 긍정적, 중립적, 부정적으로 분류하는 것입니다.",
        input=sent
    )
    # 감정 분석 결과 저장
    responses.append({
        "text": sent,
        "sentiment": response.output_text
    })

# 결과 출력
for result in responses:
    print(f"Sentiments: {result['text']}\nSentiment: {result['sentiment']}\n")

Sentiments: 나는 새로운 배트맨 영화가 좋습니다!
Sentiment: 긍정적

Sentiments: 오늘 날씨가 너무 안 좋네요.
Sentiment: 부정적

Sentiments: 커피 마시러 가는 길이 설레네요!
Sentiment: 긍정적

Sentiments: 회의가 너무 길고 지루했습니다.
Sentiment: 부정적

Sentiments: 점심으로 먹은 피자가 맛있었어요.
Sentiment: 긍정적



## In-Context Learning (ICL)

**In-Context Learning**은 모델의 가중치를 업데이트(재학습)하지 않고, **프롬프트 안에 제공한 예시(context)만으로** 모델이 작업 방식을 학습하는 기법입니다. 제공하는 예시 개수에 따라 다음과 같이 구분합니다.

| 방식 | 예시 개수 | 특징 |
| --- | --- | --- |
| **Zero-shot** | 0개 | 작업 지시만 제공. 간단한 작업에 적합 |
| **One-shot** | 1개 | 예시 1개로 출력 형식/라벨을 안내 |
| **Few-shot** | 2개 이상 | 여러 예시로 라벨 체계·형식을 명확히 학습 → 가장 안정적 |

아래에서 동일한 **고객 문의 분류** 작업을 zero-shot → one-shot → few-shot 순서로 비교해 봅니다.

### 1. Zero-shot inference (예시 없음)

- **예시를 제공하지 않고** 작업 지시(instructions)만으로 모델이 추론하게 합니다.
- 모델이 사전 학습한 지식만으로 처리하므로 간단한 작업에 적합합니다.

In [20]:
# Zero-shot inference: 예시 없이 작업 지시만으로 분류
new_inquiry = "실수로 결제했는데 주문 취소 부탁드려요."

response = client.responses.create(
    model=Model,
    instructions="당신은 고객 문의를 [배송문의, 환불요청, 상품문의, 칭찬, 기타] 중 하나의 카테고리로 분류합니다. 카테고리명만 출력하세요.",
    input=f"문의: {new_inquiry}\n카테고리:"
)

print(f"문의: {new_inquiry}")
print(f"분류 결과: {response.output_text}")

문의: 실수로 결제했는데 주문 취소 부탁드려요.
분류 결과: 환불요청


### 2. One-shot inference (예시 1개 제공)

- 작업을 설명하는 **예시를 1개** 함께 제공한 뒤 새로운 입력을 분류하게 합니다.
- 모델이 원하는 **출력 형식과 라벨**을 더 잘 따르게 됩니다.

In [22]:
# One-shot inference: 한 개의 예시를 제공하여 원하는 출력 형식을 안내
one_shot_example = (
    """
    다음은 고객 문의를 카테고리로 분류한 예시입니다.
    "문의: 주문한 옷이 아직도 안 왔어요. 언제 도착하나요?
    카테고리: 배송문의
    """
)

new_inquiry = "실수로 결제했는데 주문 취소 부탁드려요."

response = client.responses.create(
    model=Model,
    instructions="당신은 고객 문의를 [배송문의, 환불요청, 상품문의, 칭찬, 기타] 중 하나의 카테고리로 분류합니다. 카테고리명만 출력하세요.",
    input=one_shot_example + f"문의: {new_inquiry}\n카테고리:"
)

print(f"문의: {new_inquiry}")
print(f"분류 결과: {response.output_text}")

문의: 실수로 결제했는데 주문 취소 부탁드려요.
분류 결과: 환불요청


### 3. Few-shot inference (여러 예시 제공)

- 여러 개의 예시(`예시 입력 → 예시 출력`)를 함께 제공한 뒤 새로운 입력을 분류하게 합니다.
- 모델이 **라벨 체계와 출력 형식을 충분히 학습**하므로, 까다롭거나 도메인 특화된 작업에서 가장 안정적인 결과를 얻습니다.

In [23]:
# Few-shot inference: 여러 개의 예시를 제공하여 라벨 체계와 출력 형식을 명확히 학습시킴
few_shot_examples = (
    "다음은 고객 문의를 카테고리로 분류한 예시입니다.\n"
    "문의: 주문한 옷이 아직도 안 왔어요. 언제 도착하나요?\n카테고리: 배송문의\n\n"
    "문의: 사이즈가 안 맞아서 반품하고 환불받고 싶어요.\n카테고리: 환불요청\n\n"
    "문의: 이 신발은 방수가 되나요? 소재가 궁금합니다.\n카테고리: 상품문의\n\n"
    "문의: 포장도 꼼꼼하고 배송도 빨라서 정말 만족합니다!\n카테고리: 칭찬\n\n"
)

new_inquiry = "결제했는데 주문 내역에 안 보여요. 확인 부탁드려요."

response = client.responses.create(
    model=Model,
    instructions="당신은 고객 문의를 [배송문의, 환불요청, 상품문의, 칭찬, 기타] 중 하나의 카테고리로 분류합니다. 카테고리명만 출력하세요.",
    input=few_shot_examples + f"문의: {new_inquiry}\n카테고리:"
)

print(f"문의: {new_inquiry}")
print(f"분류 결과: {response.output_text}")

문의: 결제했는데 주문 내역에 안 보여요. 확인 부탁드려요.
분류 결과: 기타


## 실습: 위의 Prompt 내용을 수정해 가며 api 실행